# Document to PDF

**What it does**
Choose a document, click **Convert to PDF**, and get the PDF: the app shows where it is saved and a link to download it.

| Format | Status |
|---|---|
| Word (.docx) | works |
| Word (old .doc), PDF, Excel, Outlook (.msg) | can be chosen, not converted yet |

**Why**
Gemini and Claude read PDF, not Word. The machine in ING has Jupyter but no web page can be opened, so this runs inside the notebook: no web server, no open port.

**How to use**
1. Run all cells (Run All). The app appears below the last cell.
2. **Choose files**, then **Convert to PDF**. A spinner shows while it works.
3. Copy the path shown, or click **Download**. The PDFs are saved under `storage/converted/<date-time>/`.

**Engine**: LibreOffice when it is installed (exact Word layout); otherwise python-docx + reportlab; otherwise the builtin engine, which needs no package at all. The check cell below says which one is used.

**Needs**: `ipywidgets` only. `python-docx` and `reportlab` are optional.

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from services.config import load_env_file, settings
load_env_file()
settings.cache_clear()
from services.evidences import libreoffice
from services.presentation.converter_app import ConverterApp
print("PDFs go to:", settings().storage_root / "converted")
print("LibreOffice:", libreoffice.binary() or "not installed, the python engine is used")

## Check: what can convert Word on this machine?

Run this first. It shows which of the three ways to convert Word works here, and which one the app will use:

| Engine | Needs | Result |
|---|---|---|
| `libreoffice` | LibreOffice installed (looked for on `PATH` and in the usual folders on Linux, macOS, Windows) | exact Word layout |
| `python` | `pip install python-docx reportlab` | content kept, exact layout not |
| `builtin` | nothing (Python only) | content kept, exact layout not; Western European letters only, JPEG/PNG pictures only |

Each one is tried on a small sample document made here, so "OK" means it really works.

In [ ]:
import platform, subprocess, tempfile
from pathlib import Path
from services.evidences import libreoffice
from services.evidences.docx_builtin import sample_docx
from services.evidences.docx_converter import DocxConverter, python_engine_installed

print("Machine:", platform.system(), platform.release())
sample = Path(tempfile.mkdtemp()) / "check.docx"
sample.write_bytes(sample_docx(["Converter check"]))   # made without any package

program = libreoffice.binary()
if not program:
    print("LibreOffice: NOT FOUND. Looked on PATH (soffice, libreoffice) and in:")
    for place in libreoffice.KNOWN_PLACES:
        print("   ", place)
else:
    print("LibreOffice: found at", program)
    try:
        version = subprocess.run([program, "--version"], capture_output=True, text=True, timeout=60)
        print("   version:", (version.stdout or version.stderr).strip() or "(no answer)")
        pdf = DocxConverter("libreoffice").convert(sample, pdf=sample.with_name("lo.pdf")).pdf
        print(f"   test conversion: OK ({pdf.stat().st_size} bytes)")
    except Exception as exc:
        print("   test conversion: FAILED:", exc)
        print("   => it is there but does not work here: use ConverterApp(engine='builtin') in the last cell")

print("python-docx + reportlab:", "installed" if python_engine_installed() else "not installed")
pdf = DocxConverter("builtin").convert(sample, pdf=sample.with_name("builtin.pdf")).pdf
print(f"Builtin engine (no packages): OK ({pdf.stat().st_size} bytes)")

engine = "libreoffice" if program else "python" if python_engine_installed() else "builtin"
print()
print("=> Word files will be converted with:", engine)

In [ ]:
ConverterApp()